## 0. Описание лабораторной работы
**Задание:**

Представьте, что вы анализируйте данные для международной благотворительной организации, которая оказывает поддержку наиболее нуждающимся странам, на основе этих данных выберите страны, которым помощь нужна в первую очередь.

Вам предоставлены данные с описанием ситуации в странах, данные содержат следующую информацию:

- **name** - название страны
- **child_mort** - cмертность детей в возрасте до 5 лет на 1000 живорождений
- **exports** - экспорт товаров и услуг на душу населения. Приведено в % от ВВП на душу населения
- **health** - общие расходы на здравоохранение на душу населения. Указаны как % от ВВП на душу населения
- **Imports** - импорт товаров и услуг на душу населения. Указано в % от ВВП на душу населения.
- **Income** - чистый доход на человека
- **Inflation** - измерение годового темпа роста общего ВВП
- **life_expec** - ожидаемая продолжительность жизни
- **total_fer** - ожидаемая рождаемость
- **gdpp** - ВВП на душу населения



**Этапы выполнения:**

1. Анализ и предобработка.
    * Проанализировать данные (EDA).
    * Предобработать данные.
    * Скалировать/нормализовать данные.
2. Решите задачу с помощью следующих методов:
    * K-means/mini batch k means
    * Иерархическая кластеризация
    * DBSCAN
3. Для каждого метода определить оптимальное количество кластеров (построить график)
4. Сделайте выводы, определите, каким странам нужно помогать в первую очередь. Опишите выделите кластеры. Опишите эталонную страну в каждом кластере.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
import plotly.express as px
pio.templates.default = 'plotly_white'

import polars as pl
import numpy as np
import random
import sklearn
import os

seed = 42
random.seed(seed)
np.random.seed(seed)
sklearn.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

## 1. Работа с данными


In [ ]:
data_url = 'https://github.com/Lopa10ko/itmo-ml-2025/raw/main/lab-3/Lab3.csv'
df_pl = pl.read_csv(data_url)

print(f'Размер датасета: {df_pl.shape}\n')

Размер датасета: (167, 10)



Имеем дело с игрушечным датасетом. Сэмплов по количеству учтенных стран.

In [ ]:
display(df_pl.head())

print(f'\n\nОписательная статистика:')
display(df_pl.describe())


country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
str,f64,f64,f64,f64,i64,f64,f64,f64,i64
"""Afghanistan""",90.2,10.0,7.58,44.9,1610,9.44,56.2,5.82,553
"""Albania""",16.6,28.0,6.55,48.6,9930,4.49,76.3,1.65,4090
"""Algeria""",27.3,38.4,4.17,31.4,12900,16.1,76.5,2.89,4460
"""Angola""",119.0,62.3,2.85,42.9,5900,22.4,60.1,6.16,3530
"""Antigua and Barbuda""",10.3,45.5,6.03,58.9,19100,1.44,76.8,2.13,12200




Описательная статистика:


statistic,country,child_mort,exports,health,imports,income,inflation,life_expec,total_fer,gdpp
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""167""",167.0,167.0,167.0,167.0,167.0,167.0,167.0,167.0,167.0
"""null_count""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,38.27006,41.108976,6.815689,46.890215,17144.688623,7.781832,70.555689,2.947964,12964.155689
"""std""",null,40.328931,27.41201,2.746837,24.209589,19278.067698,10.570704,8.893172,1.513848,18328.704809
"""min""","""Afghanistan""",2.6,0.109,1.81,0.0659,609.0,-4.21,32.1,1.15,231.0
"""25%""",null,8.6,23.8,4.93,30.4,3370.0,1.85,65.3,1.8,1350.0
"""50%""",null,19.3,35.0,6.32,43.3,9960.0,5.39,73.1,2.41,4660.0
"""75%""",null,62.2,51.4,8.65,58.9,22900.0,10.9,76.8,3.91,14600.0
"""max""","""Zambia""",208.0,200.0,17.9,174.0,125000.0,104.0,82.8,7.49,105000.0


Понятно, что все численные фичи вещественные, а признак `country` является потенциально ключевым признаком (односоставным индексом).

## 2. Предобработка данных


### 2.1 Общий анализ данных

Показываем уникальные значения для представленных признаков

In [ ]:
from prettytable import PrettyTable

uniques_tb = PrettyTable()
uniques_tb.field_names = ['feature', 'n_unique']

for col in df_pl.columns:
    unique_count = df_pl[col].n_unique()
    uniques_tb.add_row([col, unique_count])

print(uniques_tb)

+------------+----------+
|  feature   | n_unique |
+------------+----------+
|  country   |   167    |
| child_mort |   139    |
|  exports   |   147    |
|   health   |   147    |
|  imports   |   151    |
|   income   |   156    |
| inflation  |   156    |
| life_expec |   127    |
| total_fer  |   138    |
|    gdpp    |   157    |
+------------+----------+


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

id_column = 'country'
key_numeric_features = [col for col in df_pl.columns if col != id_column]

fig = make_subplots(rows=3, cols=3, subplot_titles=key_numeric_features, vertical_spacing=0.1)

for i, col in enumerate(key_numeric_features):
    fig.add_trace(
        go.Box(
            y=df_pl[col],
            name=col,
            showlegend=False,
            boxpoints='outliers'
        ),
        row=(i // 3 + 1), col=(i % 3 + 1)
    )
fig.update_layout(height=1500, width=1500, showlegend=False)
fig.show()

Взглянем также на распределение каждого признака и соответсвующее ему нормальное. То есть при логнормальном распределении признака достаточно будет логарифмировать.

На самом деле это очень важный этап, так как при удалении выбросов методом IQR или Z-методом в логнормальном распределении нужно учитывать асимметрию.
А в мультимодальном такие методы просто "размажут" моды.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def diagnostic_plots(df, columns):
    subplot_titles = []
    subplot_titles
    fig = make_subplots(
        rows=len(columns), cols=2,
        subplot_titles = [f'{col} ({suffix})' for col in columns for suffix in ['original', 'log']]
    )

    for i, col in enumerate(columns):
        fig.add_trace(
            go.Histogram(x=df[col], name=col, showlegend=False),
            row=i+1, col=1
        )
        if df[col].min() > 0:
            log_data = np.log1p(df[col])
            fig.add_trace(
                go.Histogram(x=log_data, name=f'log({col})', showlegend=False),
                row=i+1, col=2
            )

    fig.update_layout(height=300*len(columns))
    fig.show()

diagnostic_plots(df_pl, key_numeric_features)

Преобразуем признаки `child_mort`, `income`, `exports`, `health`, `gdpp` прологарифмировов их и избавишись таким образом от логнормальности распределений.

In [ ]:
cols_to_log = ['child_mort', 'income', 'exports', 'health', 'gdpp']

for col in cols_to_log:
    logged_col = f'log_{col}'
    df_pl = df_pl.with_columns([
        pl.col(col).log().alias(logged_col),
    ])
    key_numeric_features.remove(col)
    key_numeric_features.append(logged_col)

## 3. Нормализация и скалирование

In [ ]:
key_numeric_features

['imports',
 'inflation',
 'life_expec',
 'total_fer',
 'log_child_mort',
 'log_income',
 'log_exports',
 'log_health',
 'log_gdpp']

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df_normed = df_pl.select(key_numeric_features).to_pandas()

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), key_numeric_features)
])
transformed_data = preprocessor.fit_transform(df_normed)


df_normed = pd.DataFrame(transformed_data, columns=key_numeric_features)
df_normed.index = df_pl[id_column]
df_normed.index.name = id_column
df_normed

,imports,inflation,life_expec,total_fer,log_child_mort,log_income,log_exports,log_health,log_gdpp
country,,,,,,,,,
Afghanistan,-0.082455,0.157336,-1.619092,1.902882,1.257285,-1.413059,-1.566676,0.450917,-1.460560
Albania,0.070837,-0.312347,0.647866,-0.859973,-0.206196,0.071749,-0.224045,0.105222,-0.122592
Algeria,-0.641762,0.789274,0.670423,-0.038404,0.223939,0.285304,0.187829,-0.963592,-0.064683
Angola,-0.165315,1.387054,-1.179234,2.128151,1.496866,-0.353135,0.818843,-1.864462,-0.221051
Antigua and Barbuda,0.497568,-0.601749,0.704258,-0.541946,-0.618844,0.605603,0.409060,-0.090571,0.608191
...,...,...,...,...,...,...,...,...,...
Vanuatu,0.240700,-0.489784,-0.852161,0.365754,0.282112,-0.918834,0.440211,-0.418445,-0.336555
Venezuela,-1.213499,3.616865,0.546361,-0.316678,-0.180538,0.486180,-0.200965,-0.576925,0.675896
Vietnam,1.380030,0.409732,0.286958,-0.661206,0.086954,-0.576020,1.007539,0.207766,-0.883883


## 4. Применение алгоритмов кластеризации

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

def create_cluster_pca(results_df, title):
    X = results_df.values

    pca = PCA(n_components=2, random_state=seed)
    X_pca = pca.fit_transform(X)
    results_df['PC1'] = X_pca[:, 0]
    results_df['PC2'] = X_pca[:, 1]

    fig = px.scatter(results_df, x='PC1', y='PC2',
                    color='cluster', title=title,
                    hover_data=[results_df.index])
    fig.update_traces(marker=dict(opacity=0.8))
    return fig

def create_cluster_tsne(df, title):
    X = df.values
    cluster_labels = df.cluster

    tsne = TSNE(n_components=3, random_state=seed,
                init='random', perplexity=min(30, (len(X) - 1) // 2))
    X_tsne_3d = tsne.fit_transform(X)
    tsne_df = pd.DataFrame({'tsne_x': X_tsne_3d[:, 0], 'tsne_y': X_tsne_3d[:, 1],
                            'tsne_z': X_tsne_3d[:, 2], 'cluster': cluster_labels,
                            'country': df.index})
    fig = go.Figure()
    unique_clusters = np.unique(cluster_labels)
    for i, cluster in enumerate(unique_clusters):
        cluster_data = tsne_df[tsne_df['cluster'] == cluster]
        fig.add_trace(go.Scatter3d(x=cluster_data['tsne_x'], y=cluster_data['tsne_y'],
                                  z=cluster_data['tsne_z'], mode='markers+text',
                                  text=cluster_data['country'], name=f'Cluster {cluster}',
                                  marker=dict(size=5, opacity=0.8)))

    fig.update_layout(title=title, width=1000, height=1000, showlegend=True)
    return fig

### 4.1. K-means

In [ ]:
from sklearn.cluster import KMeans

def kmeans_analysis(df, n_clusters=3):
    X = df.values
    countries = df.index.to_list()

    kmeans = KMeans(n_clusters=n_clusters, random_state=seed)
    labels = kmeans.fit_predict(X)
    distances = np.linalg.norm(X - kmeans.cluster_centers_[labels], axis=1)

    results_df = pd.DataFrame({
        'cluster': labels,
        'distance_to_center': distances,
        **{col: df[col].to_list() for col in df.columns}
    })
    results_df.index = df.index
    pca = create_cluster_pca(results_df, 'K-means Clustering PCA')
    tsne = create_cluster_tsne(results_df, 'K-means Clustering TSNE')

    return {'results_df': results_df, 'model': kmeans, 'pca': pca, 'tsne': tsne}

Попробуем найти оптимальный гиперапараметр количества кластеров стран по методу локтя.

In [ ]:
def kmeans_wcss_elbow(df):
    max_k = len(df) // 4
    X = df.values
    wcss = []
    k_range = range(1, max_k + 1)
    for k in k_range:
        model = KMeans(n_clusters=k, random_state=seed)
        model.fit(X)
        wcss.append(model.inertia_)
    return wcss, k_range

def plot_wcss(wcss, k_range, method):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(k_range), y=wcss,
                            mode='lines+markers', name='WCSS'))
    fig.update_layout(xaxis_title='k',
                     yaxis_title='Within-Cluster Sum of Squares',
                     title=f'Elbow method for {method}')
    fig.show()

wcss_kmeans, k_range_kmeans = kmeans_wcss_elbow(df_normed)
plot_wcss(wcss_kmeans, k_range_kmeans, 'K-means')

Явно выраженного падения значения внутрикластерного расстояния не заметил, поэтому лучше воспользуюсь автоматическим решением.

In [ ]:
!pip install kneed --quiet

In [ ]:
from kneed import KneeLocator

def automated_elbow(wcss, k_range, method):
    kl = KneeLocator(list(k_range), wcss,
                    curve='convex',
                    direction='decreasing',
                    interp_method='polynomial')

    optimal_k = kl.elbow
    print(f'Optimal number of clusters for {method}: {kl.elbow}')
    return optimal_k

optimal_k_kmeans = automated_elbow(wcss_kmeans, k_range_kmeans, 'K-means')

Optimal number of clusters for K-means: 7


Используем всего 4 кластера. Изобразим же разделение данных на 3d PCA графике.

In [ ]:
kmeans_results = kmeans_analysis(df_normed, n_clusters=optimal_k_kmeans)
kmeans_results['pca'].show()

In [ ]:
kmeans_results['tsne'].show()

В этом интерактивном графике можно залипнуть надолго, так как чудесным образом (а на самом деле это простая кластеризация) образовались кластеры Африканских стран и стран среднего востока

Далее нужно придумать критерий выделения самого "нуждающегося" кластера. Но это будем делать после получения подобных разбиений на кластеры для всех методов.

### 4.2. Иерархическая кластеризация

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import cdist

def hierarchical_wcss_elbow(X):
    max_k = len(X) // 2
    Z = linkage(X, method='ward')
    wcss = []
    k_range = range(1, max_k + 1)
    for k in k_range:
        labels = fcluster(Z, k, criterion='maxclust')
        wcss_k = 0
        for cluster_id in range(1, k + 1):
            cluster_points = X[labels == cluster_id]
            if len(cluster_points) > 0:
                cluster_center = cluster_points.mean(axis=0)
                wcss_k += np.sum(cdist(cluster_points, [cluster_center], 'sqeuclidean'))
        wcss.append(wcss_k)
    return wcss, k_range

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster

def hierarchical_analysis(df, n_clusters=3):
    X = df.values
    Z = linkage(X, method='ward')
    labels = fcluster(Z, n_clusters, criterion='maxclust') - 1

    results_df = pd.DataFrame({
        'cluster': labels,
        **{col: df[col].to_list() for col in df.columns}
    })
    results_df.index = df.index
    pca = create_cluster_pca(results_df, 'Hierarchical Clustering PCA')
    tsne = create_cluster_tsne(results_df, 'Hierarchical Clustering TSNE')
    return {'results_df': results_df, 'linkage_matrix': Z, 'pca': pca, 'tsne': tsne}

In [ ]:
wcss_hierarchical, k_range_hierarchical = hierarchical_wcss_elbow(df_normed)
plot_wcss(wcss_hierarchical, k_range_hierarchical, 'Hierarchical')
optimal_k_hierarchy = automated_elbow(wcss_hierarchical, k_range_hierarchical, 'Hierarchical')

Optimal number of clusters for Hierarchical: 13


In [ ]:
hierarchical_results = hierarchical_analysis(df_normed, n_clusters=optimal_k_hierarchy)
hierarchical_results['pca'].show()

In [ ]:
hierarchical_results['tsne'].show()

### 4.3. DBSCAN

Также воспользуемся эвристикой на гиперпараметр `min_samples` по удвоенному количеству фичей. У нас их всего 9, поэтому будем брать `min_samples=18`, а дальше уже пользоваться поиском оптимального `eps` по K-Distance Graph Method

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
from kneed import KneeLocator
import plotly.graph_objects as go

def find_eps_elbow_plotly(X, k=5):
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(X)
    distances, indices = neighbors_fit.kneighbors(X)
    k_distances = np.sort(distances[:, -1])

    kneedle = KneeLocator(range(len(k_distances)), k_distances,
                         curve='convex', direction='increasing')
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(range(len(k_distances))),
                            y=k_distances, mode='lines'))

    eps_candidate = k_distances[kneedle.elbow]
    fig.add_hline(y=eps_candidate, line_dash="dash", line_color="red",
                 annotation_text=f'Suggested eps: {eps_candidate:.3f}')

    fig.update_layout(title='K-Distance Graph',
                     xaxis_title='Points sorted by distance',
                     yaxis_title=f'{k}-NN distance')
    fig.show()
    return kneedle.elbow

dbscan_eps = find_eps_elbow_plotly(df_normed)

In [ ]:
from sklearn.cluster import DBSCAN

def dbscan_analysis(df, **kwargs):
    X = df.values
    dbscan = DBSCAN(**kwargs)
    labels = dbscan.fit_predict(X)

    results_df = pd.DataFrame({
        'cluster': labels,
        **{col: df[col].to_list() for col in df.columns}
    })
    results_df.index = df.index
    pca = create_cluster_pca(results_df, 'DBSCAN Clustering PCA')
    tsne = create_cluster_tsne(results_df, 'DBSCAN Clustering TSNE')
    return {'results_df': results_df, 'model': dbscan, 'pca': pca, 'tsne': tsne}

In [ ]:
dbscan_results = dbscan_analysis(df_normed, eps=2.57, min_samples=18)
dbscan_results['pca'].show()

In [ ]:
dbscan_results['tsne'].show()

А вот DBSCAN отработал как-то плохо, это понятно даже по визуалу получившихся кластеров.

Интересно, что при случайно подобранных гиперпараметрах алгоритм работает лучше, чем на "оптимальных". Но добиться четкой разделимости данных с помощью этого метода у меня так и не получилось.

Пробовал также и максимизировать `silhouette_score` простым гридсерчем, но так и не нашел тех гиперпараметров, которые бы явно кластеризовали данные. Самой капризным в итоге всегда оказывается параметр `eps`.

Я решил поберечь свои нервы.

In [ ]:
dbscan_analysis(df_normed, eps=0.8, min_samples=2)['tsne'].show()

## 5. Выводы


Теперь соберем в одном словаре все результаты кластеризаций (исключая DBSCAN)

In [ ]:
clustering_results = {
    'kmeans': kmeans_results,
    'hierarchical': hierarchical_results,
    # 'dbscan': dbscan_results
}

from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def evaluate_clustering_methods(df, clustering_results):
    X = df.values
    metrics = {}
    for method_name, results in clustering_results.items():
        labels = results['results_df']['cluster'].values
        metrics[method_name] = {'silhouette': silhouette_score(X, labels),
                                'calinski_harabasz': calinski_harabasz_score(X, labels),
                                'davies_bouldin': davies_bouldin_score(X, labels),
                                'n_clusters': len(np.unique(labels))}

    return pd.DataFrame(metrics).T

evaluate_clustering_methods(df_normed, clustering_results)

,silhouette,calinski_harabasz,davies_bouldin,n_clusters
kmeans,0.207814,47.224445,1.480591,7.0
hierarchical,0.206357,44.504937,1.173751,13.0


Интерпретация результатов:

1. Silhouette score (от -1 до 1, идеальное значение 1): оба метода создали слабые подструктуры. Данные плохо поддаются четкому разделению на кластеры. Разница `silhouette_score` слишком мала, чтобы говорить о преимуществе одного метода над другим
2. Calinski-Harabasz Index (неотрицательное; чем выше, тем лучше): измеряет отношение дисперсии между кластерами к дисперсии внутри кластеров. K-Means создал более плотные и отдаленные друг от друга группы (то есть более разделимые кластера)
3. Davies-Bouldin Index (неотрицательное; чем ниже, тем лучше): измеряет среднее "сходство" между кластерами, где сходство — это отношение расстояний внутри кластеров к расстоянию между кластерами. Для иерархической кластеризации сформированы более четкие границы кластеров

In [ ]:
results_df = kmeans_results['results_df']

Далее необходимо придумать, по какой логике все же ранжировать страны и кластеры стран.

Для этого посмотрим на критерии (по нашим фичам) и составим две группы таких критериев.

1) Обратные критерии (чем выше, тем страна больше нуждается в помощи)
- `log_child_mort`
- `total_fer`
- `inflation`
2) Прямые критерии (чем ниже, тем страна больше нуждается в помощи)
- `log_income`
- `log_gdpp`
- `life_expec`
- `log_health`
- `log_exports`

Далее для каждого кластера подсчитаем средние значения для обратных критериев, а среднее значение прямых критериев инвертируем (не обратное значение, а просто превращаем в `negative_*` фичи), чтобы можно было скорить кластеры "в одну сторону".

Для нуждающегося кластера значение композитного скора (`0.8 * inc_score + 0.2 * dec_score`) будет выше всего, так как это означает высокие средние показатели обратных критериев и высокие средние показатели отрицательных прямых критериев. Иначе, чем выше окажестя значения этого показателя, тем "сильнее" страны этого кластера нуждаются в гуманитарной помощи

In [ ]:
dec_indicators = ['log_child_mort', 'total_fer', 'inflation']
inc_indicators = ['log_income', 'log_gdpp', 'life_expec', 'log_health', 'log_exports']

cluster_stats = []
for cluster_id in results_df['cluster'].unique():
    cluster_data = results_df[results_df['cluster'] == cluster_id]

    dec_score = cluster_data[dec_indicators].mean().mean()
    inc_score = -cluster_data[inc_indicators].mean().mean()
    composite_score = 0.8 * inc_score + 0.2 * dec_score

    cluster_stats.append({
        'cluster': cluster_id,
        'size': len(cluster_data),
        'composite_score': composite_score,
    })

cluster_stats_df = pd.DataFrame(cluster_stats).sort_values('composite_score')
display(cluster_stats_df)
need_cluster_id = cluster_stats_df.loc[cluster_stats_df['composite_score'].idxmax(), 'cluster']

need_countries = results_df[results_df['cluster'] == need_cluster_id]
need_countries

,cluster,size,composite_score
5,2,10,-1.073801
4,1,35,-0.755803
3,6,24,-0.454479
1,0,50,0.089603
6,4,3,0.573118
2,3,17,0.910507
0,5,28,0.943588


,cluster,distance_to_center,imports,inflation,life_expec,total_fer,log_child_mort,log_income,log_exports,log_health,log_gdpp,PC1,PC2
country,,,,,,,,,,,,,
Afghanistan,5,1.151625,-0.082455,0.157336,-1.619092,1.902882,1.257285,-1.413059,-1.566676,0.450917,-1.460560,4.156200,-1.129141
Benin,5,1.555035,-0.401467,-0.654410,-0.987502,1.598107,1.436694,-1.313000,-0.435971,-1.003663,-1.249712,3.878278,-0.594630
Burkina Faso,5,0.834435,-0.716337,-0.092213,-1.427359,1.936010,1.474789,-1.509819,-0.716039,0.172906,-1.434474,4.097142,-1.026244
Burundi,5,2.096569,-0.318607,0.428709,-1.449916,2.194407,1.289276,-2.021421,-1.715710,1.458053,-2.044268,4.500874,-1.707873
Cameroon,5,1.373247,-0.824055,-0.557152,-1.495030,1.432468,1.413004,-1.003286,-0.526721,-0.473175,-0.883883,3.734322,-0.529207
Central African Republic,5,2.173201,-0.844770,-0.547664,-2.600313,1.498724,1.691253,-1.898672,-1.350844,-1.073975,-1.604350,4.769967,-1.818801
Chad,5,1.543177,-0.140457,-0.132065,-1.585257,2.413050,1.697036,-1.265106,0.132331,-0.767591,-1.137126,4.267919,-0.868197
"Congo, Dem. Rep.",5,1.956224,0.112267,1.235237,-1.472473,2.379922,1.474789,-2.206478,0.276437,0.551785,-1.797714,4.575199,-1.353285
Cote d'Ivoire,5,1.425715,-0.148743,-0.226950,-1.607814,1.538477,1.436694,-0.994133,0.547597,-0.396009,-0.931477,3.728302,-0.245052


Ну действительно, я не геополитик, но понимаю, почему эти страны в одном кластере.

Давайте для интереса посмотрим на кластер самодостаточности (с наименьшим значением показателя нуждаемости)

In [ ]:
rich_cluster_id = cluster_stats_df.loc[cluster_stats_df['composite_score'].idxmin(), 'cluster']

rich_countries = results_df[results_df['cluster'] == rich_cluster_id]
rich_countries

,cluster,distance_to_center,imports,inflation,life_expec,total_fer,log_child_mort,log_income,log_exports,log_health,log_gdpp,PC1,PC2
country,,,,,,,,,,,,,
Belgium,2,1.216570,1.152164,-0.559999,1.065167,-0.720836,-1.334808,1.231021,1.084888,1.266892,1.471980,-2.237491,2.287169
Czech Republic,2,1.618191,0.663288,-0.874070,0.783207,-0.952731,-1.577163,0.926484,0.894076,0.542791,0.931991,-1.960507,1.897935
Hungary,2,1.372853,1.226739,-0.517300,0.444855,-1.124995,-1.086072,0.732021,1.173945,0.371533,0.655784,-1.652864,1.659168
Ireland,2,0.797468,1.641040,-1.043915,1.110281,-0.594951,-1.394461,1.317604,1.474455,0.906806,1.533791,-2.347179,2.615344
Luxembourg,2,2.322326,3.940415,-0.394898,1.211786,-0.873224,-1.745035,1.885977,2.165653,0.509516,2.047517,-2.695009,3.557234
Malta,2,2.500551,4.437577,-0.374972,1.099002,-1.052114,-0.977853,0.926484,1.990462,0.763469,0.974512,-1.979685,2.671506
Netherlands,2,1.742777,0.692289,-0.657921,1.144116,-0.767215,-1.334808,1.314024,1.007539,1.518490,1.555407,-2.306740,2.255447
Singapore,2,3.826291,5.266181,-0.742749,1.369684,-1.191250,-1.745035,1.689724,2.339779,-1.085899,1.504317,-2.395627,3.554017
Slovak Republic,2,1.213240,1.280598,-0.692364,0.557639,-1.005735,-0.952790,0.831799,1.083180,0.801472,0.814118,-1.755230,1.749856


Теперь для каждой страны в кластере нуждающихся стран сформируем тоже индекс "нужды в гуманитарной помощи".

В этом случае, как и ранее, сформируем скор так, чтобы большее значение соответсвовало большему уровню "нужды в гуманитарной помощи". Для этого для обратных критериев посчитаем отношение к максимальному значению (ведь чем больше по значению эти критерии, тем страна хуже развита), а для прямых критериев будем вычитать это отношение (значения страны по показателю к максимальному значению этого атрибута) из 1, чтобы тоже максимизировать в одну сторону


In [ ]:
if 'need_score' not in need_countries.columns:
    need_countries['need_score'] = (
        0.25 * (need_countries['log_child_mort'] / need_countries['log_child_mort'].max()) +
        0.20 * (1 - need_countries['log_income'] / need_countries['log_income'].max()) +
        0.20 * (1 - need_countries['log_gdpp'] / need_countries['log_gdpp'].max()) +
        0.15 * (1 - need_countries['life_expec'] / need_countries['life_expec'].max()) +
        0.10 * (need_countries['total_fer'] / need_countries['total_fer'].max()) +
        0.10 * (need_countries['inflation'] / need_countries['inflation'].max())
    )

need_countries.sort_values('need_score', ascending=False, inplace=True)
need_countries

,cluster,distance_to_center,imports,inflation,life_expec,total_fer,log_child_mort,log_income,log_exports,log_health,log_gdpp,PC1,PC2,need_score
country,,,,,,,,,,,,,,
Senegal,5,1.255767,-0.273034,-0.562845,-0.739377,1.399340,0.997616,-1.165698,-0.377053,-0.240457,-1.064442,3.435160,-0.114166,-0.064552
Gambia,5,1.190508,-0.173601,-0.330376,-0.570201,1.830001,1.156764,-1.388099,-0.435971,-0.227944,-1.449765,3.733863,-0.459610,-0.116860
Cameroon,5,1.373247,-0.824055,-0.557152,-1.495030,1.432468,1.413004,-1.003286,-0.526721,-0.473175,-0.883883,3.734322,-0.529207,-0.125028
Cote d'Ivoire,5,1.425715,-0.148743,-0.226950,-1.607814,1.538477,1.436694,-0.994133,0.547597,-0.396009,-0.931477,3.728302,-0.245052,-0.131184
Zambia,5,1.572360,-0.662477,0.590015,-2.092785,1.624609,1.186399,-0.832293,0.139399,-0.146174,-0.811394,3.853623,-0.540624,-0.152376
Mali,5,1.330362,-0.488471,-0.323734,-1.246905,2.386548,1.618654,-1.290881,-0.491946,-0.543418,-1.295341,4.195811,-0.972506,-0.155443
Benin,5,1.555035,-0.401467,-0.654410,-0.987502,1.598107,1.436694,-1.313000,-0.435971,-1.003663,-1.249712,3.878278,-0.594630,-0.157027
Chad,5,1.543177,-0.140457,-0.132065,-1.585257,2.413050,1.697036,-1.265106,0.132331,-0.767591,-1.137126,4.267919,-0.868197,-0.172299
Kiribati,5,2.384943,1.367601,-0.594158,-1.111564,0.591023,0.942850,-1.354390,-1.194801,1.396032,-0.797793,3.218651,0.123666,-0.177269


Сенегал! Туда и отправимся с гуманитарной помощью.

Осталось поставить прививки от желтой лихорадки.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

def get_ranking_visualization(top_countries):
    fig = go.Figure()
    fig = fig.add_trace(go.Bar(y=top_countries.index, x=top_countries['need_score'],
                             orientation='h',
                             marker=dict(color=top_countries['need_score'],
                                         colorscale='reds',
                                         colorbar=dict(title='need_score'))))
    fig.update_layout(yaxis=dict(categoryorder='total ascending'),
                     xaxis_title='Need index (more - more need)',
                     height=1000)
    return fig

def get_comparison_chart(top_countries):
    fig = go.Figure()
    normalized = top_countries[['log_child_mort', 'log_income', 'log_gdpp']].copy()
    normalized['norm_log_child_mort'] = normalized['log_child_mort'] / normalized['log_child_mort'].max()
    normalized['norm_log_income'] = 1 - (normalized['log_income'] / normalized['log_income'].max())
    normalized['norm_log_gdpp'] = 1 - (normalized['log_gdpp'] / normalized['log_gdpp'].max())

    fig.add_trace(go.Bar(name='norm_log_child_mort', orientation='h',
                        y=top_countries.index, x=normalized['norm_log_child_mort']))
    fig.add_trace(go.Bar(name='norm_log_income', orientation='h',
                        y=top_countries.index, x=normalized['norm_log_income']))
    fig.add_trace(go.Bar(name='norm_log_gdpp', orientation='h',
                        y=top_countries.index, x=normalized['log_gdpp']))
    fig.update_layout(barmode='group', height=1000, yaxis=dict(categoryorder='total ascending'))
    return fig

In [ ]:
get_ranking_visualization(need_countries).show()

In [ ]:
get_comparison_chart(need_countries).show()

Вот и все, конец!

По щеке катится скупая мужская слеза (опять).

Если хотите понаблюдать, как <s>плачут мужчины</s>, смотрите отчеты по лабороторной 1 и 2.

Небольшое саммари:

**Лабораторная работа выполнена в рамках курса "Машинное обучение" ИТМО, 2025**